# 0. API Key 로드

In [1]:
from dotenv import load_dotenv


load_dotenv()

True

# 1. RAG system 구현

- 거짓말을 줄이고 최신 정보를 활용하기 위해 쓰임

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 문서 로드
data_path = "../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf"
docs = PyMuPDFLoader(data_path).load()

# 2. 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 100)  # 600자 내에 100자가 포함되어 있음
split_docs = splitter.split_documents(docs)

# 3. 임베딩 객체 정의(수치화 준비 단계)
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

# 4. 벡터스토어에 저장(수치화 진행 후 벡터 저장)
vectorstore = Chroma.from_documents(
  documents = split_docs,
  embedding = embedding,
  persist_directory="./Chroma",
  collection_name="sesac"
)

# 5. 검색
retriever = vectorstore.as_retriever()  # Chroma 에서 제공해주는 검색기 생성
result = retriever.invoke("AI 클로벌 거버넌스 행동계획을 발표한 나라는 어느나라아?")

print(type(result))
print(len(result))
result[0]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


<class 'list'>
4


Document(metadata={'author': 'dj', 'creationDate': "D:20250909164309+09'00'", 'creationdate': '2025-09-09T16:43:09+09:00', 'creator': 'Hwp 2022 12.0.0.4204', 'file_path': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 'format': 'PDF 1.6', 'keywords': '', 'modDate': "D:20250909164309+09'00'", 'moddate': '2025-09-09T16:43:09+09:00', 'page': 4, 'producer': 'Hancom PDF 1.3.0.550', 'source': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 'subject': '', 'title': '', 'total_pages': 28, 'trapped': ''}, page_content='* 주로 남반구나 북반구의 저위도에 위치한 120여개 개발도상국을 지칭\n∙AI 기술의 돌파구를 마련하기 위한 협력에 더욱 집중하여 기초 과학기술 R&D 협력을 심화하고 기술과 \n인재 교류를 강화하는 한편, 글로벌 거버넌스에 더욱 집중하기 위한 세계 AI 협력기구 설립을 제안\n£ 세계 각국의 공동 참여를 촉구하는 ‘AI 글로벌 거버넌스 행동계획’ 발표  \nn 중국 정부는 이번 행사에서 글로벌 AI 개발과 거버넌스에서 아래와 같은 조치를 통해 세계 각국의 \n참여와 협력을 촉구하는 ‘AI 글로벌 거버넌스 행동계획’도 발표\n∙(AI 기회 공동 포착과 혁신 발전) 다양한 국제 과학기술 협력 플랫폼의 구축과 혁신 친화적 정책 환경 조성, \n정책과 규제 조율 강화, “AI 플러스” 개방형 응용 시나리오의 심도 있는 탐색을 추진\n∙(AI를 통한 산업 발전) AI를 통해 제조, 소비, 유통, 의료, 교육, 농업 등 다양한 분야의 역량을 강화하고 \n지능형 인프라 구축

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 6. 최종 프롬프트 완성하기
system_prompt = "당신은 유능한 AI assistant입니다. 친절하게 답변해주세요."
user_prompt = """
아래의 사용자 질의에 대하여 context를 참고하여 답변해주세요.

--사용자 질의--
{question}

--context--
{context}
"""

# 최종 프롬프트
final_prompt = ChatPromptTemplate([
  ("system", system_prompt),
  ("user", user_prompt)
])

# 모델 정의
model = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)
# 체인 생성
chain = final_prompt|model|StrOutputParser() 

# 응답 생성
response=chain.invoke({"question": "AI 클로벌 거버넌스 행동계획을 발표한 나라는 어느나라아?", "context": result[0].page_content})
print(response)

AI 글로벌 거버넌스 행동계획을 발표한 나라는 **중국**입니다. 중국 정부는 이번 행사에서 글로벌 AI 개발과 거버넌스에 대한 참여와 협력을 촉구하는 행동계획을 발표했습니다.


## 1.1 Data Load

- 문서 로드 
  -인공지능 산업의 최신 동향 pdf
- 문서 포맷: pdf
- 문서 출처: https://spri.kr/posts/view/23908?code=AI-Brief&s_year=&data_page=1

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

data_path = "../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf"
docs = PyMuPDFLoader(data_path).load()

print(type(docs))
print(len(docs))
docs[0]

<class 'list'>
28


Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2022 12.0.0.4204', 'creationdate': '2025-09-09T16:43:09+09:00', 'source': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 'file_path': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 'total_pages': 28, 'format': 'PDF 1.6', 'title': '', 'author': 'dj', 'subject': '', 'keywords': '', 'moddate': '2025-09-09T16:43:09+09:00', 'trapped': '', 'modDate': "D:20250909164309+09'00'", 'creationDate': "D:20250909164309+09'00'", 'page': 0}, page_content='2025년\n9월호\n인공지능 산업의 최신 동향')

In [ ]:
"""
Document(
    metadata={
        'producer': 'Hancom PDF 1.3.0.550', 
        'creator': 'Hwp 2022 12.0.0.4204', 
        'creationdate': '2025-09-09T16:43:09+09:00', 
        'source': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 
        'file_path': '../data/SPRi AI Brief_9월호_산업동향_0909_F.pdf', 
        'total_pages': 28, 
        'format': 'PDF 1.6', 
        'title': '', 
        'author': 'dj', 
        'subject': '', 
        'keywords': '', 
        'moddate': '2025-09-09T16:43:09+09:00', 
        'trapped': '', 
        'modDate': "D:20250909164309+09'00'", 
        'creationDate': "D:20250909164309+09'00'", 
        'page': 0
    }, 
    page_content='2025년\n9월호\n인공지능 산업의 최신 동향'
)
"""

In [7]:
# 표지, 목차 등 필요없는 페이지 제외
docs = docs[3:-2] # 3번 인덱스 ~ -3인덱스까지만 포함(= 앞의 3페이지, 맨뒤 2페이지 제외)

print(len(docs))
print(docs[0].metadata["page"])

23
3


In [10]:
print(len(docs[0].page_content))
print(docs[0].page_content)

1847
SPRi AI Brief
2025년 9월호
2
영국 과학혁신기술부, 영국 컴퓨트 로드맵 발표
n 영국 과학혁신기술부는 AI의 잠재력을 실현할 수 있도록 국가 컴퓨팅 생태계를 혁신하기 위한 
장기 계획으로서 ‘영국 컴퓨트 로드맵’을 수립
n 이 로드맵은 공공 연산(Compute) 인프라 현대화와 컴퓨팅을 활용한 혁신 촉진, 영국의 AI 
선도를 위한 AI 인프라 구축, 자주적이고 안전하며 지속 가능한 역량 확보를 추진
KEY Contents
£ 영국의 AI 리더십 확보를 위한 공공 연산 인프라 현대화와 AI 인프라 구축 추진
n 영국 과학혁신기술부(DSIT)가 2025년 7월 17일 AI의 막대한 잠재력을 실현할 컴퓨팅 인프라를 구축해 
영국의 AI 리더십을 확고히 하기 위한 ‘영국 컴퓨트 로드맵(UK Compute Roadmap)’을 발표*
* 기술적 컴퓨터 연산 인프라 확대에 초점을 두어, 일반적 용어인 컴퓨팅(Computing) 대신 컴퓨트 용어를 채택
∙경제 전반에 걸쳐 혁신과 성장, 기회를 위한 플랫폼을 제공하는 세계적 수준의 컴퓨팅 생태계를 구축한다는 
비전에 따라, 최우선 과제와 가장 혁신적인 기회에 컴퓨팅을 집중하는 성과 중심적 생태계를 조성할 계획
n (컴퓨트 생태계 구축) 영국 정부는 2대의 신규 AI 슈퍼컴퓨터를 기반으로 2030년까지 공공 연산 
인프라 현대화에 최대 20억 파운드(한화 약 3조 7,300억 원)를 투입할 계획 
∙2030년까지 ‘AI 연구 자원(AI Research Resource, AIRR)’*을 20배 늘리는데 10억 파운드 이상을 
투자하고, 에든버러의 신규 국가 슈퍼컴퓨터 서비스 구축에 최대 7억 5천만 파운드를 투입
* AI 연구와 혁신 가속화를 위해 연구자와 대학, 중소기업 등에 AI 슈퍼컴퓨팅 자원을 제공하는 국가 인프라
∙유사한 성격의 외국 정부 및 컴퓨팅 센터와 협력하여 영국에서 자체적으로 조달하기 어려운 연구 인프라 
접근성을 확보하고 협업과 기술·지식 교류를 활성화하며 글로벌 AI 환경에서 영국

In [11]:
# 소제목을 제외시키기 위해 페이지별로 글자수 확인해보기
for page in docs:
  print(len(page.page_content))

1847
1726
1636
1825
1605
19
1545
1789
1077
1850
1689
1803
1302
19
1458
1883
1372
1804
19
1102
1738
1691
1638


In [13]:
# 19글자에 해당하는 소제목 페이지는 제외시키기
docs = [page for page in docs if len(page.page_content) != 19]
print(len(docs))
print()

for page in docs:
  print(len(page.page_content))

20

1847
1726
1636
1825
1605
1545
1789
1077
1850
1689
1803
1302
1458
1883
1372
1804
1102
1738
1691
1638


## 1.2 Split Document

- 동일한 주제 및 소재를 다루는 단락 단위로 split
- 문서 분석을 하여 보통 몇 글자 단위로 비슷한 내용을 담고있는지(동일한 주제 및 소재를 다루는지) 파악 후 분리 
- 정확하게 분리되지 않을 경우 overlap을 적절히 부여 
- 또는 동일 주제 및 소재를 다루는 chunk는 동일한 메타데이터 부여 

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# splitter 객체 정의
splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 100)

# 문서 분할
split_docs = splitter.split_documents(docs)

# 확인
print(type(split_docs))
print(len(split_docs))
print() # 공백
print(type(split_docs[0]))
print(len(split_docs[0].page_content)) # 600자 이전에 \n을 찾아서 끊음, 무조건 600자를 맞추지 않음
print(split_docs[0].page_content)

<class 'list'>
72

<class 'langchain_core.documents.base.Document'>
544
SPRi AI Brief
2025년 9월호
2
영국 과학혁신기술부, 영국 컴퓨트 로드맵 발표
n 영국 과학혁신기술부는 AI의 잠재력을 실현할 수 있도록 국가 컴퓨팅 생태계를 혁신하기 위한 
장기 계획으로서 ‘영국 컴퓨트 로드맵’을 수립
n 이 로드맵은 공공 연산(Compute) 인프라 현대화와 컴퓨팅을 활용한 혁신 촉진, 영국의 AI 
선도를 위한 AI 인프라 구축, 자주적이고 안전하며 지속 가능한 역량 확보를 추진
KEY Contents
£ 영국의 AI 리더십 확보를 위한 공공 연산 인프라 현대화와 AI 인프라 구축 추진
n 영국 과학혁신기술부(DSIT)가 2025년 7월 17일 AI의 막대한 잠재력을 실현할 컴퓨팅 인프라를 구축해 
영국의 AI 리더십을 확고히 하기 위한 ‘영국 컴퓨트 로드맵(UK Compute Roadmap)’을 발표*
* 기술적 컴퓨터 연산 인프라 확대에 초점을 두어, 일반적 용어인 컴퓨팅(Computing) 대신 컴퓨트 용어를 채택
∙경제 전반에 걸쳐 혁신과 성장, 기회를 위한 플랫폼을 제공하는 세계적 수준의 컴퓨팅 생태계를 구축한다는


## 1.3 Embedding(수치화)

- 차원이 높을 수록 정교한 표현 가능 (절대적이지 않고 문서마다 적절하게 잘 표현되는 차원을 찾아야함)
- 어떤 임베딩 알고리즘을 쓰느냐에 따라 사용자 입력값과 유사도를 계산했을 때 유사도가 다르게 나올 수 있음 
- OpenAI 임베딩 모델
    - 다국어 지원이 되는 모델 중 가장 성능이 무난해서 많이 쓰임 
    - 좋은 모델 같은 경우는 모델 사이즈가 커서 GPU환경에서 돌려야할 수도 있음, 
        - OpenAI의 임베딩 모델을 쓰는 경우엔 OpenAI의 자원을 빌려쓰기 때문에 무거운 모델도 일반 노트북에서 돌릴 수 있음 
    - 하지만 비용 발생 
    - https://platform.openai.com/docs/guides/embeddings#embedding-models

In [24]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 객체 정의
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 확인용(임베딩 시 어떤 값을 반환하는지 확인)
embedded_document = embedding.embed_documents([split_docs[0].page_content])

In [30]:
print("원문 데이터 타입: ", type(split_docs[0].page_content), "\n")
print("embedded_document 데이터 타입: ", type(embedded_document), "\n")
print("첫 번쨰 요소의 데이터 타입: ", type(embedded_document[0]), "\n")
print("embedded_doccument의 요소 개수: ", len(embedded_document), "\n") # 임베딩 된 청크 개수
print("embedded_doccument의 첫 번째 요소 길이: ", len(embedded_document[0]), "\n")  # 1536: 임베딩 모델의 차원 수 (모델이 표현할 수 있는 단어의 수)
print("첫 번째 토큰의 벡터값 확인:\n", embedded_document[0])

원문 데이터 타입:  <class 'str'> 

embedded_document 데이터 타입:  <class 'list'> 

첫 번쨰 요소의 데이터 타입:  <class 'list'> 

embedded_doccument의 요소 개수:  1 

embedded_doccument의 첫 번째 요소 길이:  1536 

첫 번째 토큰의 벡터값 확인:
 [0.052127886563539505, 0.019013311713933945, -0.013087663799524307, 0.011450313031673431, 0.04250427708029747, -0.03459598496556282, -0.015181689523160458, 0.06683062016963959, -0.0097628403455019, -0.02528425119817257, 0.027266893535852432, -0.04428642615675926, -0.051103148609399796, -0.016685377806425095, 0.0006366312736645341, -0.045600760728120804, -0.06665240973234177, -0.010336469858884811, -0.005920079071074724, 0.007891582325100899, -0.02791292406618595, -0.045890361070632935, 0.028380736708641052, -0.0030129472725093365, -0.002543740440160036, -0.04047707840800285, 0.033237095922231674, -0.020205125212669373, 0.01227455772459507, -0.014168092049658298, 0.004126790910959244, -0.0325242355465889, 0.019592510536313057, -0.03116534650325775, 0.0025465250946581364, -0.004486005287617445,

## 1.4 Vectorstore

- Chroma
  - 대표적인 로컬 DB
  - 오픈소스 벡터 데이터베이스
  - 임베딩을 저장하고 검색할 수 있도록 설계된 데이터베이스
  - 벡터 검색 및 정보 검색과 같은 작업에 사용 됨

In [49]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
  documents=split_docs,
  embedding=embedding,
  persist_directory="./Chroma",
  collection_name="sesac2"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [32]:
# .get(): vectorstore에 저장되어있는 chunk ID, 임베딩값, 원문 등의 정보를 확인
store_info=vectorstore.get()
store_info

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['6e693d43-bf53-431a-9d43-9c3677f46714',
  'db1d6676-f5ec-4193-a835-7032fa3d57ec',
  'd7099781-d71d-410e-abab-d940fca527ff',
  '7b247802-a113-4428-9ded-739c3fdbe94e',
  'f0dd23e1-eacd-47b3-9176-263c08db6329',
  '0c1afd6d-3c47-4cba-9221-2bd9c7b9b879',
  '25e94108-4362-4cec-89b8-ef53ad3b80d2',
  'c198ca60-3f60-4b1f-a644-ec6ea54c1dd1',
  'df8aa2ad-31db-44d3-a9c6-9b8096681c06',
  'cd7ffbd5-f10e-4a93-8b89-f811d5105eee',
  '4cb142ed-d9b8-4ac6-b519-0c1313223fc7',
  '731843aa-83a8-43f5-8013-1d78a4ee5b00',
  'e61814a0-2e0e-463d-8f79-f6bf6ed5fe2e',
  '4bc984bd-c6f7-41a4-ae3d-d1936e165ee3',
  'ce3d3ed1-26ae-471a-be73-3b40206c28b8',
  'd3512be6-aa05-441a-8b85-a641e639b57e',
  '3372836d-81f5-4ee7-94ae-5580c34c01ab',
  '46d5c5bf-873b-4187-b17b-0326463e8c57',
  '8e2be98f-3e73-49f1-a17f-af0f33758ee7',
  '1d4a1dce-fb74-4c39-b382-8a2fe4c62d12',
  'b00d60e1-44ce-48a0-8fbe-ec0614a464be',
  'caea1070-7bb3-4dc4-91a1-81dbbabf8012',
  'd9663f3a-b796-4f07-be53-6e4819efe0fe',
  '08977906-17d6-4345-b627-

In [37]:
store_info.keys()

# 청크 개수 확인
print(len(store_info["ids"]))

# documents 확인
store_info["documents"]

72


['SPRi AI Brief\n2025년 9월호\n2\n영국 과학혁신기술부, 영국 컴퓨트 로드맵 발표\nn 영국 과학혁신기술부는 AI의 잠재력을 실현할 수 있도록 국가 컴퓨팅 생태계를 혁신하기 위한 \n장기 계획으로서 ‘영국 컴퓨트 로드맵’을 수립\nn 이 로드맵은 공공 연산(Compute) 인프라 현대화와 컴퓨팅을 활용한 혁신 촉진, 영국의 AI \n선도를 위한 AI 인프라 구축, 자주적이고 안전하며 지속 가능한 역량 확보를 추진\nKEY Contents\n£ 영국의 AI 리더십 확보를 위한 공공 연산 인프라 현대화와 AI 인프라 구축 추진\nn 영국 과학혁신기술부(DSIT)가 2025년 7월 17일 AI의 막대한 잠재력을 실현할 컴퓨팅 인프라를 구축해 \n영국의 AI 리더십을 확고히 하기 위한 ‘영국 컴퓨트 로드맵(UK Compute Roadmap)’을 발표*\n* 기술적 컴퓨터 연산 인프라 확대에 초점을 두어, 일반적 용어인 컴퓨팅(Computing) 대신 컴퓨트 용어를 채택\n∙경제 전반에 걸쳐 혁신과 성장, 기회를 위한 플랫폼을 제공하는 세계적 수준의 컴퓨팅 생태계를 구축한다는',
 '∙경제 전반에 걸쳐 혁신과 성장, 기회를 위한 플랫폼을 제공하는 세계적 수준의 컴퓨팅 생태계를 구축한다는 \n비전에 따라, 최우선 과제와 가장 혁신적인 기회에 컴퓨팅을 집중하는 성과 중심적 생태계를 조성할 계획\nn (컴퓨트 생태계 구축) 영국 정부는 2대의 신규 AI 슈퍼컴퓨터를 기반으로 2030년까지 공공 연산 \n인프라 현대화에 최대 20억 파운드(한화 약 3조 7,300억 원)를 투입할 계획 \n∙2030년까지 ‘AI 연구 자원(AI Research Resource, AIRR)’*을 20배 늘리는데 10억 파운드 이상을 \n투자하고, 에든버러의 신규 국가 슈퍼컴퓨터 서비스 구축에 최대 7억 5천만 파운드를 투입\n* AI 연구와 혁신 가속화를 위해 연구자와 대학, 중소기업 등에 AI 슈퍼컴퓨팅 자원을 제공하는 국가 인프라\n∙유사한 성격의 외국 정부 및 컴퓨팅 센

In [39]:
# vector -> 비어져 있음: embeddings는 기본적으로 생략되어 반환되지 않음
store_info["embeddings"]

In [40]:
# 벡터값 확인 위해서는 별도로 벡터값을 불러오도록 지정할 필요가 있음
store_info = vectorstore.get(include=["embeddings"])

store_info["embeddings"]

array([[ 0.05219522,  0.0190439 , -0.01307109, ..., -0.03035435,
         0.0130488 ,  0.01166704],
       [ 0.04056756,  0.02868559,  0.04678655, ..., -0.01784355,
         0.01108915,  0.00281862],
       [ 0.0327722 ,  0.03158558,  0.04802562, ..., -0.03149928,
         0.01695783,  0.0028317 ],
       ...,
       [ 0.02931511,  0.01118953, -0.00399669, ..., -0.01434338,
         0.02127944,  0.00813839],
       [ 0.02655829,  0.02088481, -0.01263919, ..., -0.00734722,
         0.0260414 ,  0.00487661],
       [-0.00702073,  0.00592223, -0.03156907, ...,  0.01037373,
         0.01582085, -0.03064912]])

In [41]:
# 문서 추가

# 1. 추가할 문서 로드
data_path = "../data/SPRi AI Brief_10월호_산업동향_1002_F.pdf"
additional_docs = PyMuPDFLoader(data_path).load()

# 2. 문서 분할 & page = 4만 추출
split_additional_docs = splitter.split_documents(additional_docs)
page_4_docs = [doc for doc in split_additional_docs if doc.metadata["page"] == 4]

# 3. vectorstore에 10월호 4페이지에 해당되는 청크만 추가해보기
vectorstore.add_documents(documents=page_4_docs)

# 4. store_info 업데이트
store_info = vectorstore.get()

# 문서 추가 후 청크 개수 확인
print(len(store_info["ids"]))

75


In [45]:
store_info["ids"][-3:]

['7639e65e-8d41-4a3f-be3a-6e65c0ba00dd',
 'a3d5afef-1383-4705-9046-b5e5b8569b93',
 '072abe41-7ae9-407c-968a-080a43b82f24']

In [47]:
# 측정 청크 삭제( 맨 마지막에 추가한 청크를 다시 삭제해보자)

vectorstore.delete(ids=store_info["ids"][-3:])

# 삭제 후 청크 개수 확인
vectorstore._collection.count()

Delete of nonexisting embedding ID: 7639e65e-8d41-4a3f-be3a-6e65c0ba00dd
Delete of nonexisting embedding ID: a3d5afef-1383-4705-9046-b5e5b8569b93
Delete of nonexisting embedding ID: 072abe41-7ae9-407c-968a-080a43b82f24
Delete of nonexisting embedding ID: 7639e65e-8d41-4a3f-be3a-6e65c0ba00dd
Delete of nonexisting embedding ID: a3d5afef-1383-4705-9046-b5e5b8569b93
Delete of nonexisting embedding ID: 072abe41-7ae9-407c-968a-080a43b82f24
Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


72

In [48]:
# collection의 모든 데이터 삭제 (실행 후 꼭 주석 처리 해주기 !!!)
# vectorstore.delete(ids=store_info["ids"]) 
# vectorstore._collection.count()

Delete of nonexisting embedding ID: 7639e65e-8d41-4a3f-be3a-6e65c0ba00dd
Delete of nonexisting embedding ID: a3d5afef-1383-4705-9046-b5e5b8569b93
Delete of nonexisting embedding ID: 072abe41-7ae9-407c-968a-080a43b82f24
Delete of nonexisting embedding ID: 7639e65e-8d41-4a3f-be3a-6e65c0ba00dd
Delete of nonexisting embedding ID: a3d5afef-1383-4705-9046-b5e5b8569b93
Delete of nonexisting embedding ID: 072abe41-7ae9-407c-968a-080a43b82f24
Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


0

## 1.5 Retriever

- as_retriever(): 
    - 벡터 DB객체를 효율적인 검색기로 사용할 수 있도록 변환해주는 역할 
    - as_retirever를 사용하면 쿼리 최적화 기법들이 적용되고 langchain 생태계와의 연동성이 높아지며 유사도 검색 전후에 필터나 제약조건 등을 추가로 적용할 수 있어서 일반적인 검색기보다 더 성능이 좋음 
    - 파라미터
        - search_type: str, 검색 유형 정의 (default=similarity, mmr, similarity_score_threshold)
        - search_kwargs: dict, 검색 함수에 전달할 추가 인자
            - k: 반환할 문서 수 (default: 4)
            - score_threshold: 최소 유사도 임계값
                - 쿼리가 문서와 얼마나 유사한지의 정도
                - 1이면 완벽하게 일치함을 의미
            - fetch_k: MMR 알고리즘에 전달할 문서 수 (default:20)
                - 20이면 먼저 20개를 가져온 후 mmr로 다양성 및 유사도를 계산 후 순위가 높은 개수(k값)를 골라 옴
            - lambda_mult: MMR 결과의 다양성 조절 (default: 0.5, 0~1)
                - 1에 가까운 숫자면 다양성을 고려하여 다양한 문서 반환
                - 0에 가까우면 유사도만 고려하여 반환
            - filter: 문서 메타데이터 필터링 

[결론]
- 표준 유사성 검색기(similarity)는 유사도가 높은 문서 상위 K개를 빠르게 검색하므로 관련성이 높은 문서를 빠른 속도로 검색할 때 좋고 
- Mmr 유사성 검색기는 검색의 중복을 줄이고 다양성을 포함해야할 때 사용하는 것이 좋음(다만, 표준 검색기에 비해 약간 더 느림) 

## 1.5.1 기본값 적용

- similarity 기반 검색

In [50]:
# 검색기 객체 생성
retriever = vectorstore.as_retriever()

# 검색 진행
results = retriever.invoke("AI 글로벌 거버넌스 행동계획을 발표한 나라는 어느 나라야?")

print(type(results))
print(len(results))

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


<class 'list'>
4


In [51]:
for chunk in results:
  print(chunk.page_content)
  print("\n", "-"*20)

* 주로 남반구나 북반구의 저위도에 위치한 120여개 개발도상국을 지칭
∙AI 기술의 돌파구를 마련하기 위한 협력에 더욱 집중하여 기초 과학기술 R&D 협력을 심화하고 기술과 
인재 교류를 강화하는 한편, 글로벌 거버넌스에 더욱 집중하기 위한 세계 AI 협력기구 설립을 제안
£ 세계 각국의 공동 참여를 촉구하는 ‘AI 글로벌 거버넌스 행동계획’ 발표  
n 중국 정부는 이번 행사에서 글로벌 AI 개발과 거버넌스에서 아래와 같은 조치를 통해 세계 각국의 
참여와 협력을 촉구하는 ‘AI 글로벌 거버넌스 행동계획’도 발표
∙(AI 기회 공동 포착과 혁신 발전) 다양한 국제 과학기술 협력 플랫폼의 구축과 혁신 친화적 정책 환경 조성, 
정책과 규제 조율 강화, “AI 플러스” 개방형 응용 시나리오의 심도 있는 탐색을 추진
∙(AI를 통한 산업 발전) AI를 통해 제조, 소비, 유통, 의료, 교육, 농업 등 다양한 분야의 역량을 강화하고 
지능형 인프라 구축과 공유, 국경 간 AI 응용 협력, 모범사례 공유 등으로 실물경제의 발전을 촉진
∙(디지털 인프라 구축 가속화) 전 세계 청정에너지, 지능형 컴퓨팅 자원, 데이터센터와 같은 인프라 구축을

 --------------------
정책･법제
기업･산업
기술･연구
인력･교육
3
중국 정부, 글로벌 거버넌스를 위한 세계 AI 협력기구 창설 제안
n 중국의 리창 총리가 궁극적으로 인류에게 이익이 되는 AI를 위한 세계 각국의 공동 거버넌스를 
강조하며 세계 AI 협력 기구의 설립을 제안 
n 중국 정부는 AI 기회의 공동 포착과 혁신 발전, AI를 통한 산업 발전, 표준과 규범 합의, AI 역량 
강화를 위한 국제협력을 골자로 하는 ‘AI 글로벌 거버넌스 행동계획’도 발표
KEY Contents
£ 중국, 세계 AI 협력기구 설립 통해 AI 글로벌 거버넌스 주도 의지 피력
n 리창(李强) 중국 총리가 2025년 7월 26일 상하이에서 열린 2025년 세계AI대회(WAIC) 및 AI 글로벌 
거버넌스 고위급 회의 기조연설에

## 1.5.2 옵션 변경

- mmr 기반 검색

In [52]:
retriever_mmr = vectorstore.as_retriever(
  search_type = "mmr",
  search_kwargs ={
    "k":2,                # 최종적으로 잔환할 청크 개수
    "fetch_k": 10,        # 우선적으로 선별할 청크 수(먼저 10개 선별 후 다양성 및 유사도 계산)
    "lambda_mult": 0.25   # 다양성 조절(0~1, 1에 가까울수록 중복 없이 다양한 문서 반환)
  }
)

# 검색 진행
results_mmr = retriever_mmr.invoke("AI 글로벌 거버넌스 행동계획을 발표한 나라는 어느 나라야?")

print(type(results_mmr))
print(len(results_mmr))

<class 'list'>
2


In [53]:
for chunk in results_mmr:
  print(chunk.page_content)
  print("\n", "-"*20)

* 주로 남반구나 북반구의 저위도에 위치한 120여개 개발도상국을 지칭
∙AI 기술의 돌파구를 마련하기 위한 협력에 더욱 집중하여 기초 과학기술 R&D 협력을 심화하고 기술과 
인재 교류를 강화하는 한편, 글로벌 거버넌스에 더욱 집중하기 위한 세계 AI 협력기구 설립을 제안
£ 세계 각국의 공동 참여를 촉구하는 ‘AI 글로벌 거버넌스 행동계획’ 발표  
n 중국 정부는 이번 행사에서 글로벌 AI 개발과 거버넌스에서 아래와 같은 조치를 통해 세계 각국의 
참여와 협력을 촉구하는 ‘AI 글로벌 거버넌스 행동계획’도 발표
∙(AI 기회 공동 포착과 혁신 발전) 다양한 국제 과학기술 협력 플랫폼의 구축과 혁신 친화적 정책 환경 조성, 
정책과 규제 조율 강화, “AI 플러스” 개방형 응용 시나리오의 심도 있는 탐색을 추진
∙(AI를 통한 산업 발전) AI를 통해 제조, 소비, 유통, 의료, 교육, 농업 등 다양한 분야의 역량을 강화하고 
지능형 인프라 구축과 공유, 국경 간 AI 응용 협력, 모범사례 공유 등으로 실물경제의 발전을 촉진
∙(디지털 인프라 구축 가속화) 전 세계 청정에너지, 지능형 컴퓨팅 자원, 데이터센터와 같은 인프라 구축을

 --------------------
SPRi AI Brief
2025년 9월호
12
애플, 6천억 달러 규모의 ‘미국 제조 프로그램’ 발표
n 애플이 미국 내에서 애플 제품에 사용되는 주요 부품을 생산하는 ‘미국 제조 프로그램’에 
총 6천억 달러를 투자하고 향후 4년간 핵심 영역에서 2만 명의 직원을 직접 고용할 계획
n 애플은 미국 내 실리콘 공급망을 통해 2025년에 190억 개 이상의 애플 제품용 칩을 생산할 
계획이며, ‘애플 인텔리전스’의 기반이 되는 애플 서버도 2026년부터 미국에서 양산 예정
KEY Contents
£ 다수 공급업체와 협력해 미국 내 애플 제품용 부품과 실리콘 공급망 구축 계획
n 애플(Apple)이 2025년 8월 7일 미국 내 공급망과 첨단 제조 관련 투자 총액을 6천억 달러로 확대하는 


# 2. 최종 프롬프트 완성

In [54]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system_prompt = "당신은 유능한 AI assistant입니다. 친절하게 답변해주세요~"
user_prompt = """\

사용자 질의를 보고 context를 참고하여 답변해주세요.

--사용자 질의--
{question}

--context--
{context}

"""

final_prompt = ChatPromptTemplate([
  ("system", system_prompt),
  ("user", user_prompt)
])

model = ChatOpenAI(model = "gpt-4o-mini", temperature=0)

chain = final_prompt | model | StrOutputParser()

context_text = "\n".join(chunk.page_content for chunk in results)
response = chain.invoke({"question": "AI 글로벌 거버넌스 행동계획을 발표한 나라는 어느 나라야?", "context": context_text})
print(response)

AI 글로벌 거버넌스 행동계획을 발표한 나라는 **중국**입니다. 중국 정부는 2025년 세계 AI 대회에서 AI 글로벌 거버넌스 행동계획을 발표하며, 세계 각국의 참여와 협력을 촉구했습니다. 이 계획은 AI 기회의 공동 포착, 산업 발전, 디지털 인프라 구축, 표준 제정 및 국제 협력을 포함하고 있습니다.
